# UnconBacktest Example: TS Momentum in Cryptocurrencies

We will run a simple "uncon backtest" to test whether 4 of the larger coins (BTC, ETH, ADA, BNB) show time-series momentum. In other words, if they have done well recently, do they tend to continue to do well?

1. Run the first cell to compute returns for these coins since 2016.
2. The signal on each day for each coin will be: sqrt(10)*(avg past 10 day ret - avg past 365 day ret) / (stdev past 365 day ret). You can think of this as the z-score of the past 10 day returns, and it tells us if a coin is doing better than it usually does. Technical note: we multiply by sqrt(10) here because the stdev is calculated on daily data but we want to "z-score" the average 10 day returns.
3. The signal from (2) will have extreme values. Pass them through a tanh function to curtail these. These are your final daily portfolio weights.
4. Compute the returns to the weights from (3). 
5. What is the Sharpe of these strategies? Plot their cumulative returns.
6. How correlated are the timing strategies with the underlying assets?

In [4]:
# compute returns 
import yfinance as yf 
import numpy as np

data = yf.download(['ADA-USD','BNB-USD','BTC-USD','ETH-USD'],'2016-1-1')
### data = yf.download(['ETH-USD'],'2016-1-1')
ret = data['Close'] / data['Close'].shift() - 1
###pickle.dump(ret, open("returns.pkl", "wb"))

[*********************100%***********************]  4 of 4 completed


In [8]:
import sys
import pickle
print(sys.executable)
pickle.dump(ret, open("homework.006.returns.pkl", "wb"))

/Users/waynekirkschmidt/anaconda3/envs/wsqwks/bin/python


In [9]:
import sys
sys.version
returns = ret

In [10]:
# compute returns 
import yfinance as yf 
import numpy as np
yeardays = 365
data = yf.download(['ADA-USD','BNB-USD','BTC-USD','ETH-USD'],'2016-1-1')
returns = data['Close'] / data['Close'].shift() - 1

[*********************100%***********************]  4 of 4 completed


In [11]:
yeardays = 365
port = np.sqrt(10)*(returns.rolling(10,min_periods=1).mean() - returns.rolling(yeardays,min_periods=10).mean()) 
port = port / returns.rolling(yeardays,min_periods=10).std()
port = np.tanh(port)
strategic_returns = port.shift()*returns 

In [12]:
yeardays = 365
strategic_returns.mean()/strategic_returns.std()*np.sqrt(yeardays)

Ticker
ADA-USD    0.616421
BNB-USD    0.768997
BTC-USD    0.588992
ETH-USD    0.701844
dtype: float64

In [14]:
import matplotlib
strategic_returns.cumsum().plot()

ModuleNotFoundError: No module named 'matplotlib'

In [15]:
strategic_returns.corrwith(ret)

Ticker
ADA-USD    0.069840
BNB-USD    0.148885
BTC-USD   -0.121142
ETH-USD   -0.148082
dtype: float64